# System1Bench — Colab T4
从公开 fork 克隆并运行 Laya 评测，无需 GH_TOKEN。
先选择 T4 GPU，再从上到下执行。结果保存到 Colab 临时磁盘，完成后下载 ZIP。

复用 Colab 当前 Python 和预装 PyTorch，不下载或重装 PyTorch/CUDA。
直接使用 Colab 当前 Python，一次安装 Laya 0.3.20、Transformers 5.16.1 和项目依赖。
安装时锁定预装 torch 版本；安装后检查所需模块，随后依次运行 English 和 Multilingual，记录失败并继续。
本 notebook 尚未在 Colab 实机验证，T4 结果应与原始 A100 参考环境分开报告。

每个模型独立进程运行，case 的 audit/predict/GPU 同步/decode 异常保留在错误清单中；模型崩溃后仍执行下一个模型。


In [1]:
# 1. 复用 Colab 当前 Python 和预装 PyTorch
import os, sys, json, subprocess
from pathlib import Path
from importlib.metadata import version

assert Path("/content").exists(), "请在 Google Colab 中运行"
assert sys.version_info >= (3, 12), "需要 Python 3.12 或更新版本"
LOG_DIR = Path("/content/system1bench-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_NUMBER = 0

def run(args, cwd=None):
    global COMMAND_NUMBER
    COMMAND_NUMBER += 1
    args = list(map(str, args))
    log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    while log.exists():
        COMMAND_NUMBER += 1
        log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    print("+", " ".join(args), flush=True)
    print("日志:", log, flush=True)
    lines = []
    with log.open("w", encoding="utf-8") as stream:
        stream.write("+ " + " ".join(args) + "\n")
        stream.flush()
        proc = subprocess.Popen(
            args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
                lines.append(line)
            code = proc.wait()
        except BaseException:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise
    if code:
        tail = "".join(lines[-40:])
        raise RuntimeError(
            f"命令退出码 {code}；完整日志：{log}\n最后 40 行：\n{tail}")


run(["nvidia-smi"])
import torch
assert torch.cuda.is_available(), "请选择 GPU 运行时"
TORCH_VERSION = version("torch")
print("复用 PyTorch:", TORCH_VERSION, "GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/system1bench")
if not REPO.exists():
    run(["git", "clone", "--depth", "1",
         "https://github.com/Jaffe2718/system1bench.git", REPO])
assert (REPO / "requirements-colab-t4.txt").is_file(), "现有克隆过旧，请在新运行时重新克隆本 fork"
run(["git", "rev-parse", "HEAD"], cwd=REPO)

PY = Path(sys.executable)
CONSTRAINT = Path("/content/system1bench-torch-constraint.txt")
CONSTRAINT.write_text("torch==" + TORCH_VERSION)
run([PY, "--version"])


+ nvidia-smi
日志: /content/system1bench-logs/command-001.log
Mon Oct 26 09:01:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |            

In [2]:
# 2. 一次安装项目依赖；复用 Colab 预装 PyTorch
# 不创建虚拟环境、不重复预解析、不升级已经满足要求的包。
# torch 版本被约束为预装版本；若依赖不兼容会停止，而不会换版本。
run([PY, "-m", "pip", "install", "--disable-pip-version-check",
     "--constraint", CONSTRAINT, "-r", "requirements-colab-t4.txt"], cwd=REPO)
run([PY, "-c",
     "from importlib.metadata import version; import sys; "
     "assert version('torch') == sys.argv[1]; print('PyTorch preserved:', version('torch'))",
     TORCH_VERSION])


+ /usr/bin/python3 -m pip install --disable-pip-version-check --constraint /content/system1bench-torch-constraint.txt -r requirements-colab-t4.txt
日志: /content/system1bench-logs/command-005.log
Obtaining file:///content/system1bench
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 33.2 MB/s  0:00:00
  Building editable for system1bench (pyproject.toml): started
  Building editable for system1bench (pyproject.toml): finished with status 'done'
  Created wheel for system1bench: filename=system

In [3]:
# 3. 导入检查：覆盖数据下载、准备、模型评测和报告所需模块
run([PY, "-c", """
import importlib, sys, traceback
modules = [
    'numpy', 'pyarrow', 'pyarrow.parquet', 'huggingface_hub',
    'yaml', 'requests', 'torch', 'transformers', 'laya',
    'safetensors', 'tokenizers',
    'transformers.models.modernbert.modeling_modernbert',
    'system1bench.download', 'system1bench.prepare',
    'system1bench.laya_adapter', 'system1bench.run',
    'system1bench.metrics', 'system1bench.report', 'system1bench.colab_batch',
]
failed_imports = {}
for name in modules:
    try:
        importlib.import_module(name)
        print('IMPORT OK:', name, flush=True)
    except Exception as exc:
        failed_imports[name] = type(exc).__name__ + ': ' + str(exc)
        traceback.print_exc()
print('FAILED IMPORTS:', failed_imports, flush=True)
import torch, transformers
assert sys.version_info >= (3, 12)
assert torch.cuda.is_available(), 'CUDA 不可用，请选择 GPU 运行时'
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
"""], cwd=REPO)

+ /usr/bin/python3 -c 
import importlib, sys, traceback
modules = [
    'numpy', 'pyarrow', 'pyarrow.parquet', 'huggingface_hub',
    'yaml', 'requests', 'torch', 'transformers', 'laya',
    'safetensors', 'tokenizers',
    'transformers.models.modernbert.modeling_modernbert',
    'system1bench.download', 'system1bench.prepare',
    'system1bench.laya_adapter', 'system1bench.run',
    'system1bench.metrics', 'system1bench.report', 'system1bench.colab_batch',
]
failed_imports = {}
for name in modules:
    try:
        importlib.import_module(name)
        print('IMPORT OK:', name, flush=True)
    except Exception as exc:
        failed_imports[name] = type(exc).__name__ + ': ' + str(exc)
        traceback.print_exc()
print('FAILED IMPORTS:', failed_imports, flush=True)
import torch, transformers
assert sys.version_info >= (3, 12)
assert torch.cuda.is_available(), 'CUDA 不可用，请选择 GPU 运行时'
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capabi

In [4]:
# 4. 两个模型依次运行，结果仅保存到 Colab 临时磁盘
from datetime import datetime, timezone
from uuid import uuid4

MODELS = ["english", "multilingual"]
assert MODELS and all(m in ("english", "multilingual") for m in MODELS)
SESSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUTPUT = Path("/content/system1bench-results") / SESSION
OUTPUT.mkdir(parents=True, exist_ok=True)
print("模型:", MODELS, "结果目录:", OUTPUT)
print("会话结束前请下载结果 ZIP。")

freeze = subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
(OUTPUT / "environment-pip-freeze.txt").write_text(freeze)
(OUTPUT / "repository-commit.txt").write_text(commit)


模型: ['english', 'multilingual'] 结果目录: /content/system1bench-results/20261026-090258-57d45d
会话结束前请下载结果 ZIP。


40

In [5]:
# 5. 固定版本数据下载与准备
# 子进程日志会实时显示并写入 /content/system1bench-logs。
# 失败时查看异常中最后 40 行的实际原因，不要跳过哈希校验。
run([PY, "-u", "-m", "system1bench.download"], cwd=REPO)
run([PY, "-u", "-m", "system1bench.prepare"], cwd=REPO)


+ /usr/bin/python3 -u -m system1bench.download
日志: /content/system1bench-logs/command-008.log
Verified ag_news
Verified emotion
Verified banking77
Verified boolq
Verified sst5
Verified xnli_en
Verified xnli_zh
Verified massive_en
Verified massive_zh
Verified prompt_injections
Verified typed_decisions
Verified jevbench
Verified reflexbench
Verified jev_laya
Not fetched: clinc/api_metadata.json (historical API receipt; not a dataset file or evaluation input)
Verified clinc
Verified turtle
Verified aegis
+ /usr/bin/python3 -u -m system1bench.prepare
日志: /content/system1bench-logs/command-009.log
Frozen 16 suites; 14516 requests/model; MASSIVE aligned: True
SHA256 15b7d88a842a53f2e9737931275b4399b5998c053070997ccd1f8a9fa808b138
Frozen 36 suites 22934 requests/model


下面运行完整评测：每个模型 36 个套件、26,450 个决策，batch size 固定为 1 以隔离单个 case。
不会添加单独的 smoke 推理。case 出错后记录失败；模型启动失败或进程崩溃后继续下一个模型。
完成不代表全部成功：检查最终的 OK / DONE_WITH_ERRORS / FAILED，以及报告生成状态。
状态表仅描述代码执行是否成功，不代表预测是否正确。需要中止时可使用 Colab 的停止按钮。


In [6]:
# 6. 一次运行全部指定模型；失败信息会保存，后续模型继续
run([PY, "-u", "-m", "system1bench.colab_batch",
     "--models", *MODELS, "--output", OUTPUT], cwd=REPO)


+ /usr/bin/python3 -u -m system1bench.colab_batch --models english multilingual --output /content/system1bench-results/20261026-090258-57d45d
日志: /content/system1bench-logs/command-010.log
START_MODEL english

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 5 files: 100%|██████████| 5/5 [00:04<00:00,  1.01it/s]
/content/system1bench/system1bench/laya_adapter.py:32: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  self.agent = Agent(str(checkpoint), subfolder=sub or None, device="cuda")
DONE english ag_news {'sha256': '5cfe62e53d35f20786acd685aec9c49c7a02ae13be1f555f9544cb0f4ec44b3b', 'requests': 1000, 'decisions': 1000, 'errors': 0, 'inference_seconds': 35.86527857000067}
DONE english 

In [7]:
# 7. 汇总执行状态；完整失败清单保存在 failures.csv
import csv
status = json.loads((OUTPUT / "run_status.json").read_text())
print("总状态:", status["status"])
for model, result in status["models"].items():
    print(model, result["status"], "失败决策:", result.get("errors", "见日志"),
          "报告:", {k: v["status"] for k, v in result.get("reports", {}).items()})
print("\n每个套件的运行状态:")
print((OUTPUT / "run_status.csv").read_text())
with (OUTPUT / "failures.csv").open() as stream:
    failures = list(csv.DictReader(stream))
print("失败记录数:", len(failures))
for failure in failures[:100]:
    print(failure)
print("完整失败清单:", OUTPUT / "failures.csv")
print("各模型报告与原始结果:", OUTPUT / "runs")


总状态: OK
english OK 失败决策: 0 报告: {'metrics': 'OK', 'report': 'OK'}
multilingual OK 失败决策: 0 报告: {'metrics': 'OK', 'report': 'OK'}

每个套件的运行状态:
model,suite,status,decisions,valid_decisions,errors
english,ag_news,OK,1000,1000,0
english,emotion,OK,1000,1000,0
english,banking77,OK,1000,1000,0
english,boolq,OK,1000,1000,0
english,boolq_choice,OK,1000,1000,0
english,sst5,OK,1000,1000,0
english,sst5_choice,OK,1000,1000,0
english,xnli_en,OK,1000,1000,0
english,xnli_zh,OK,1000,1000,0
english,massive_en,OK,1000,1000,0
english,massive_zh,OK,1000,1000,0
english,prompt_injections,OK,116,116,0
english,typed_decisions,OK,2000,2000,0
english,jevbench_original,OK,72,72,0
english,jevbench_easy,OK,48,48,0
english,jevbench_hard,OK,111,111,0
english,reflexbench_reflex-public-choice-v1,OK,95,95,0
english,jev_laya_triage,OK,501,501,0
english,jev_laya_moderation,OK,426,426,0
english,jev_laya_routing,OK,411,411,0
english,jev_laya_claims,OK,300,300,0
english,jev_laya_reviews,OK,300,300,0
english,jev_laya_guard,OK,2

In [8]:
# 8. 保存依赖版本和本次使用的仓库提交，打包下载
import shutil
from google.colab import files
freeze = subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
(OUTPUT / "environment-pip-freeze.txt").write_text(freeze)
(OUTPUT / "repository-commit.txt").write_text(commit)
shutil.copytree(LOG_DIR, OUTPUT / "logs", dirs_exist_ok=True)
archive = shutil.make_archive("/content/system1bench-t4-results", "zip", OUTPUT)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>